# VMBench workshop

Classify contest videos as **real (0)** or **generated (1)**, then explore motion diagnostics.

Components: **PAS** amplitude · **OIS** human structure · **TCS** coherence · **CAS** learned plausibility.

In [ ]:
from pathlib import Path
if not Path("/content/workshop/.git").exists():
    !git clone --recursive https://github.com/KK-one/vmbench-workshop.git /content/workshop
%cd /content/workshop

## 0. Initialization

Prepare dependencies and checkpoints on a T4 GPU.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "utils.py").is_file():
    PROJECT_DIR = Path("/content/workshop")
sys.path.insert(0, str(PROJECT_DIR))
from utils import (initialize, unpack_contest_archive, browse_contest, write_binary_submission,
                   compute_pas, compute_ois, compute_tcs, compute_cas)

WORK_DIR = Path("/content")
DATASET_ZIP = WORK_DIR / "contest_vids.zip"
VIDEO_PATH = WORK_DIR / "contest_vids" / "d713b9.mp4"
workshop = initialize(PROJECT_DIR, metrics=["PAS", "OIS", "TCS", "CAS"], work_dir=WORK_DIR)


## 1. Download and upload the dataset

[Join the competition](https://www.kaggle.com/t/ac5704b939234a8d96a93d8aabc66bde), accept its rules, and download the videos ZIP from **Data** (or use **Download All**).

In Colab, open **Files → Upload** and place the ZIP in `/content`. Wait for the upload to finish, then run the next cell. Nested ZIPs are supported; video bytes are preserved.


In [ ]:
if not DATASET_ZIP.is_file():
    archives = sorted(WORK_DIR.glob("*.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Upload the dataset ZIP to /content, or set DATASET_ZIP in cell 0.")
    DATASET_ZIP = archives[0]

CONTEST_DIR = WORK_DIR / "contest_vids"
CONTEST_VIDEO_PATHS = unpack_contest_archive(DATASET_ZIP, CONTEST_DIR)
print(f"Contest: {len(CONTEST_VIDEO_PATHS)} videos | {CONTEST_DIR}")
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"Example video is missing: {VIDEO_PATH}")
workshop.input_paths = [VIDEO_PATH]
workshop.select_videos()


## 2. Browse and create a baseline

The examples below use `d713b9.mp4`. The all-real baseline covers every contest video.


In [ ]:
video_browser = browse_contest([VIDEO_PATH] + [p for p in CONTEST_VIDEO_PATHS if p != VIDEO_PATH])
baseline_path = write_binary_submission(CONTEST_VIDEO_PATHS, WORK_DIR / "baseline_all_real.csv")
from IPython.display import FileLink, display
display(FileLink(str(baseline_path)))
# Download when ready:
# from google.colab import files
# files.download(str(baseline_path))

## PAS

Motion amplitude. `subject` is an English detection prompt before SAM.


In [ ]:
pas = compute_pas(VIDEO_PATH, subject="person", box_threshold=0.25,
                  text_threshold=0.20, grid_size=30)


## OIS

Human structural integrity. N/A when no person or pose is detected.


In [ ]:
ois = compute_ois(VIDEO_PATH, bbox_thr=0.3, nms_thr=0.3)


## TCS

Object persistence. Check detection warnings when interpreting high scores.


In [ ]:
tcs = compute_tcs(VIDEO_PATH, subject="person", box_threshold=0.25,
                  text_threshold=0.20, grid_size=50)


## CAS

Learned plausibility. Defaults use 10 segments × 3 crops; changing sampling can change scores.


In [ ]:
cas = compute_cas(VIDEO_PATH, test_num_segment=10, test_num_crop=3, num_workers=1)


## Results

Collect the scores above without running evaluators again. Missing scores are not zeros; inspect status and runtime.

In [ ]:
workshop.show_metric_summary(VIDEO_PATH)
results_df = workshop.evaluate_dataset(recompute=False)
workshop.plot_results(results_df)


## Submit to Kaggle

Download `baseline_all_real.csv` and edit its `label` column, or fill `PREDICTIONS` below for every contest ID. Upload the CSV to Kaggle manually. Accuracy = correct labels / all videos; a perfect submission scores 1.0.

In [ ]:
PREDICTIONS = {}  # Example format: {"a1b2c3": 0, "d4e5f6": 1}; use actual contest IDs.
if PREDICTIONS:
    submission_path = write_binary_submission(CONTEST_VIDEO_PATHS, WORK_DIR / "submission.csv",
                                              predictions=PREDICTIONS)
    display(FileLink(str(submission_path)))
else:
    print("Edit the baseline CSV or fill PREDICTIONS for every contest video.")